# 学習後の 4B を 64 フレームで評価する（学習なし、Colab A100）

GRPO 3 回目のモデル（`grpo3_time/step_00400`）は、長い動画で当たらなくても外れた距離が 40 秒 → 16 秒に縮んでいる。32 枚で 20 分を見る間隔（37.5 秒）が壁になっている可能性があるので、64 枚（間隔 18.75 秒、許容幅 9.4 秒）で測り直す。
比較のため学習前の 4B も 64 枚で測る。詳細は `docs/grpo3_time.md`。

前提: Drive の `MyDrive/movie_llm/` に `frames_f64_s640.zip` と `grpo3_time/step_00400/`（LoRA）がある。1 本 722 問で約 45 分、2 本で約 1.5 時間（64 枚は 1 問 4 秒前後）。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. リポジトリと依存パッケージ

In [ ]:
%cd /content
!rm -rf movie_llm && git clone -q https://github.com/kanaharu20/movie_llm.git
%cd /content/movie_llm
!pip install -q "transformers>=5.0" "accelerate>=1.0" "peft>=0.15" "av>=14" "pillow>=10" "pandas>=2" "pyarrow" "huggingface_hub" "python-dotenv" "openai"
!pip uninstall -y -q torchao
!pip install -q -e . --no-deps

## 2. Drive と 64 フレーム

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/movie_llm'
ADAPTER = f'{DRIVE}/grpo3_time/step_00400'
import os
!mkdir -p /content/frames
!unzip -q -o {DRIVE}/frames_f64_s640.zip -d /content/frames
os.makedirs('/content/videos', exist_ok=True)
for vid in os.listdir('/content/frames'):
    open(f'/content/videos/{vid}.mp4', 'a').close()
print('フレーム', len(os.listdir('/content/frames')), '本', 'LoRA', os.path.exists(ADAPTER))

## 3. 学習後の 4B、64 フレーム（722 問、約 45 分）

In [ ]:
%cd /content/movie_llm
!python scripts/jvqa_predict.py --name grpo3_time-4b_f64 --task time --model Qwen/Qwen3-VL-4B-Instruct --frames 64 --max-new-tokens 96 \
    --adapter {ADAPTER} --no-fetch --video-dir /content/videos --frames-dir /content/frames --out-dir {DRIVE}/results
!python scripts/jvqa_eval_time.py --predictions {DRIVE}/results/grpo3_time-4b_f64/predictions.jsonl --name grpo3_time-4b_f64 --out-root {DRIVE}/results

## 4. 学習前の 4B、64 フレーム（比較用、約 45 分）

In [ ]:
%cd /content/movie_llm
!python scripts/jvqa_predict.py --name time-4b_f64 --task time --model Qwen/Qwen3-VL-4B-Instruct --frames 64 --max-new-tokens 96 \
    --no-fetch --video-dir /content/videos --frames-dir /content/frames --out-dir {DRIVE}/results
!python scripts/jvqa_eval_time.py --predictions {DRIVE}/results/time-4b_f64/predictions.jsonl --name time-4b_f64 --out-root {DRIVE}/results

終わったら、手順 3 と 4 の表（全体・short・medium）を貼り、`predictions.jsonl` を手元の `results/grpo3_time-4b_f64/` と `results/time-4b_f64/` に置く。